# MIST-HER2: direct flow with Phikon and stain supervision

Start from H&E plus small Gaussian noise and predict HER2 with Phikon conditioning. The active `stable` recipe uses FM, LPIPS, coarse L1, DAB and range losses; critic and contrastive losses are disabled. `INITIALIZE_FROM=None` starts from scratch.


In [ ]:
MODE = "train"                  # smoke = CPU-проверка с random feature backbones
DATASET_OVERRIDE = "/kaggle/input/datasets/vasiliygorelov/mist-her2"          # None: поиск kaggle_bundle.json в Add Input
RESUME_FROM = None               # strict resume только этой v2 версии
INITIALIZE_FROM = None  # EMA из текущего run; None = с нуля
SESSION_STEPS = 2000
IMAGE_SIZE = 256                 # native crop512 → resize256; 512 = дороже/больше деталей
TRAIN_PAIRS = None               # None: весь train; 1024: group-balanced subset
MICRO_BATCH = 1                  # effective batch4; 2 если хватает VRAM
FM_PATH = "linear"              # linear / vp — обе готовые реализации TorchCFM
LOSS_RECIPE = "stable"           # текущий контроль: FM+LPIPS+coarse L1+DAB; critic/ASP/NCE off
PHIKON_PATH = None               # offline папка pretrained Phikon, иначе Input/download

## Environment and source provenance

Setup installs extra packages in a private Kaggle directory. Reused author modules retain their repository, commit and SHA records. Attach the prepared MIST bundle and use a fresh kernel.


In [ ]:
import os,sys,json,stat,zipfile,tempfile,subprocess,importlib.metadata,importlib.util,shutil
from pathlib import Path
INPUT_ROOT=Path(os.environ.get("IHC_INPUT_ROOT","/kaggle/input"))
WORK_ROOT=Path(os.environ.get("IHC_WORK_ROOT","/kaggle/working"));WORK_ROOT.mkdir(parents=True,exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR",str(WORK_ROOT/".matplotlib"))
os.environ["USE_TF"]="0";os.environ["USE_FLAX"]="0"
for backend in ("JAX","TENSORFLOW","CUPY"):os.environ[f"POT_BACKEND_DISABLE_{backend}"]="1"
if DATASET_OVERRIDE:
    search_root=Path(DATASET_OVERRIDE).expanduser()
    if search_root.is_file() and search_root.name=="kaggle_bundle.json":
        BUNDLE=search_root.parent
    elif (search_root/"kaggle_bundle.json").is_file():
        BUNDLE=search_root
    else:
        candidates=sorted(search_root.rglob("kaggle_bundle.json")) if search_root.is_dir() else []
        if len(candidates)!=1:
            found="\n".join(str(p.parent) for p in candidates) or "не найдено"
            raise FileNotFoundError(f"В DATASET_OVERRIDE нужен один подготовленный MIST bundle. Найдено: {found}. Нужны code/, data/ и manifests/.")
        BUNDLE=candidates[0].parent
else:
    markers=list(INPUT_ROOT.rglob("kaggle_bundle.json"))
    if not markers:
        archives=list(INPUT_ROOT.rglob("mist-her2-flow-kaggle.zip"))
        if len(archives)==1:
            unpack=Path(tempfile.gettempdir())/"paper-fm-dataset"
            with zipfile.ZipFile(archives[0]) as archive:
                for member in archive.infolist():
                    if not (unpack/member.filename).resolve().is_relative_to(unpack.resolve()) or stat.S_ISLNK(member.external_attr>>16):raise ValueError("Unsafe ZIP")
                archive.extractall(unpack)
            markers=list(unpack.rglob("kaggle_bundle.json"))
    if len(markers)!=1:raise FileNotFoundError("Подключи один MIST bundle или укажи DATASET_OVERRIDE")
    BUNDLE=markers[0].parent
if not (BUNDLE/"kaggle_bundle.json").exists():raise FileNotFoundError("Нет kaggle_bundle.json")
DATA_ROOT=BUNDLE/"data/HER2";MANIFEST=BUNDLE/"manifests/group-disjoint/pairs.jsonl"
sys.path.insert(0,str(BUNDLE/"code"))
PINS={"torchcfm":"1.0.7","POT":"0.9.6.post1","diffusers":"0.35.2","torchdiffeq":"0.2.5",
      "ema-pytorch":"0.7.7","lpips":"0.1.4","transformers":"4.57.6",
      "huggingface-hub":"0.36.2","tokenizers":"0.22.2"}
def version(name):
    try:return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:return None
DEPS=WORK_ROOT/"paper-fm-deps"
if any(version(name)!=value for name,value in PINS.items()):
    command=[sys.executable,"-m","pip","install","--quiet","--no-deps","--upgrade","--target",str(DEPS)]
    subprocess.run(command+[f"{name}=={value}" for name,value in PINS.items()],check=True)
    sys.path.insert(0,str(DEPS));importlib.invalidate_caches()
if importlib.util.find_spec("safetensors") is None:
    subprocess.run([sys.executable,"-m","pip","install","--quiet","--no-deps","--target",str(DEPS),"safetensors>=0.4"],check=True)
    sys.path.insert(0,str(DEPS));importlib.invalidate_caches()
print("Pinned APIs:",PINS)

In [ ]:
# Авторские modules сохраняются как обычные читаемые .py; не обучающие runtime patches.
VENDOR = {'provenance.json': '[\n  {\n    "file": "asp_loss.py",\n    "repository": "lifangda01/AdaptiveSupervisedPatchNCE",\n    "commit": "b113ed287efaea9b848c8e1859d04072cb6eb449",\n    "original_path": "models/asp_loss.py",\n    "sha256": "9321ddbe9611d1c13ccc236bc3cc415296301bbe1a963d7c610e3fe29839a939",\n    "modified": false\n  },\n  {\n    "file": "patchnce.py",\n    "repository": "lifangda01/AdaptiveSupervisedPatchNCE",\n    "commit": "b113ed287efaea9b848c8e1859d04072cb6eb449",\n    "original_path": "models/patchnce.py",\n    "sha256": "2bd50b34679786219a4d5c5858a3a277b0fd4467cc49b05d37a2041cb5f7f678",\n    "modified": false\n  },\n  {\n    "file": "uni_losses.py",\n    "repository": "facevoid/UNIStainNet",\n    "commit": "bf3576bef1a0ab08821b4f7c8db35026f62892d3",\n    "original_path": "src/models/losses.py",\n    "sha256": "544ff9434fadc665298a1395034964eeea296c4816521c77f87f38b6cbd4c1ce",\n    "modified": false\n  },\n  {\n    "file": "uni_discriminator.py",\n    "repository": "facevoid/UNIStainNet",\n    "commit": "bf3576bef1a0ab08821b4f7c8db35026f62892d3",\n    "original_path": "src/models/discriminator.py",\n    "sha256": "2589a54cc39434f419c5d244230d7f77e285e60177401a874c3557b492a6aeea",\n    "modified": false\n  },\n  {\n    "file": "uni_dab.py",\n    "repository": "facevoid/UNIStainNet",\n    "commit": "bf3576bef1a0ab08821b4f7c8db35026f62892d3",\n    "original_path": "src/utils/dab.py",\n    "sha256": "0b7102da249a2796dc324232b537a231fd586f70d8d9aff8746fc5f16ae49d66",\n    "modified": false\n  },\n  {\n    "file": "uni_methods.py",\n    "repository": "facevoid/UNIStainNet",\n    "commit": "bf3576bef1a0ab08821b4f7c8db35026f62892d3",\n    "original_path": "src/models/trainer.py",\n    "extraction": "compute_dab_intensity_loss and compute_he_edge_loss; bodies unchanged",\n    "sha256": "bd841d5be7ee54cc3096b64f8b638430aa9f520e0c87074515fe7377ac4db75b",\n    "modified": "AST extraction only"\n  }\n]\n', 'patchnce.py': 'from packaging import version\nimport torch\nfrom torch import nn\n\n\nclass PatchNCELoss(nn.Module):\n    def __init__(self, opt):\n        super().__init__()\n        self.opt = opt\n        self.cross_entropy_loss = torch.nn.CrossEntropyLoss(reduction=\'none\')\n        self.mask_dtype = torch.uint8 if version.parse(torch.__version__) < version.parse(\'1.2.0\') else torch.bool\n\n    def forward(self, feat_q, feat_k):\n        num_patches = feat_q.shape[0]\n        dim = feat_q.shape[1]\n        feat_k = feat_k.detach()\n\n        # pos logit\n        l_pos = torch.bmm(\n            feat_q.view(num_patches, 1, -1), feat_k.view(num_patches, -1, 1))\n        l_pos = l_pos.view(num_patches, 1)\n\n        # neg logit\n\n        # Should the negatives from the other samples of a minibatch be utilized?\n        # In CUT and FastCUT, we found that it\'s best to only include negatives\n        # from the same image. Therefore, we set\n        # --nce_includes_all_negatives_from_minibatch as False\n        # However, for single-image translation, the minibatch consists of\n        # crops from the "same" high-resolution image.\n        # Therefore, we will include the negatives from the entire minibatch.\n        if self.opt.nce_includes_all_negatives_from_minibatch:\n            # reshape features as if they are all negatives of minibatch of size 1.\n            batch_dim_for_bmm = 1\n        else:\n            batch_dim_for_bmm = self.opt.batch_size\n\n        # reshape features to batch size\n        feat_q = feat_q.view(batch_dim_for_bmm, -1, dim)\n        feat_k = feat_k.view(batch_dim_for_bmm, -1, dim)\n        npatches = feat_q.size(1)\n        l_neg_curbatch = torch.bmm(feat_q, feat_k.transpose(2, 1))\n\n        # diagonal entries are similarity between same features, and hence meaningless.\n        # just fill the diagonal with very small number, which is exp(-10) and almost zero\n        diagonal = torch.eye(npatches, device=feat_q.device, dtype=self.mask_dtype)[None, :, :]\n        l_neg_curbatch.masked_fill_(diagonal, -10.0)\n        l_neg = l_neg_curbatch.view(-1, npatches)\n\n        out = torch.cat((l_pos, l_neg), dim=1) / self.opt.nce_T\n\n        loss = self.cross_entropy_loss(out, torch.zeros(out.size(0), dtype=torch.long,\n                                                        device=feat_q.device))\n\n        return loss\n', '__init__.py': '', 'uni_discriminator.py': '"""\nPatchGAN discriminator for HER2 image realism scoring.\n\nSupports both unconditional (3ch HER2 only) and conditional (6ch H&E+HER2) modes.\nReturns patch-level logits AND intermediate features for feature matching loss.\n\nArchitecture:\n    C64(SN) -> C128(SN+IN) -> C256(SN+IN) -> C512(SN+IN,s1) -> 1ch(SN,s1)\n    70x70 receptive field, output [B, 1, 30, 30] for 512x512 input.\n    ~2.8M params (3ch) or ~2.8M params (6ch).\n\nReferences:\n  - Isola et al., "Image-to-Image Translation with Conditional Adversarial Networks" (CVPR 2017)\n  - Miyato et al., "Spectral Normalization for GANs" (ICLR 2018)\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.autograd as autograd\nfrom torch.nn.utils import spectral_norm\n\n\nclass PatchDiscriminator(nn.Module):\n    """PatchGAN discriminator with spectral normalization.\n\n    Returns both logits and intermediate features (for feature matching loss).\n\n    Args:\n        in_channels: 3 for unconditional (HER2 only), 6 for conditional (H&E + HER2)\n        ndf: base number of discriminator filters\n        n_layers: number of intermediate conv layers\n    """\n\n    def __init__(self, in_channels=3, ndf=64, n_layers=3):\n        super().__init__()\n        self.n_layers = n_layers\n\n        # Build layers as a list (not sequential) so we can extract features\n        self.layers = nn.ModuleList()\n\n        # First layer: spectral norm, no instance norm\n        self.layers.append(nn.Sequential(\n            spectral_norm(nn.Conv2d(in_channels, ndf, 4, stride=2, padding=1)),\n            nn.LeakyReLU(0.2, inplace=True),\n        ))\n\n        # Intermediate layers: spectral norm + instance norm\n        nf_mult = 1\n        for n in range(1, n_layers):\n            nf_mult_prev = nf_mult\n            nf_mult = min(2 ** n, 8)\n            self.layers.append(nn.Sequential(\n                spectral_norm(nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, 4, stride=2, padding=1)),\n                nn.InstanceNorm2d(ndf * nf_mult),\n                nn.LeakyReLU(0.2, inplace=True),\n            ))\n\n        # Penultimate layer: stride 1\n        nf_mult_prev = nf_mult\n        nf_mult = min(2 ** n_layers, 8)\n        self.layers.append(nn.Sequential(\n            spectral_norm(nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, 4, stride=1, padding=1)),\n            nn.InstanceNorm2d(ndf * nf_mult),\n            nn.LeakyReLU(0.2, inplace=True),\n        ))\n\n        # Final layer: 1-channel output, no activation (hinge loss uses raw logits)\n        self.layers.append(nn.Sequential(\n            spectral_norm(nn.Conv2d(ndf * nf_mult, 1, 4, stride=1, padding=1)),\n        ))\n\n    def forward(self, x, return_features=False):\n        """\n        Args:\n            x: [B, C, H, W] in [-1, 1]. C=3 (unconditional) or C=6 (conditional).\n            return_features: if True, also return intermediate features for FM loss.\n\n        Returns:\n            logits: [B, 1, H\', W\'] patch-level real/fake logits\n            features: list of intermediate feature maps (only if return_features=True)\n        """\n        features = []\n        h = x\n        for layer in self.layers:\n            h = layer(h)\n            if return_features:\n                features.append(h)\n\n        if return_features:\n            return h, features\n        return h\n\n\n# ======================================================================\n# Loss functions\n# ======================================================================\n\ndef hinge_loss_d(d_real, d_fake):\n    """Discriminator hinge loss."""\n    return (torch.relu(1.0 - d_real).mean() + torch.relu(1.0 + d_fake).mean()) / 2\n\n\ndef hinge_loss_g(d_fake):\n    """Generator hinge loss."""\n    return -d_fake.mean()\n\n\ndef r1_gradient_penalty(discriminator, real_images, weight=10.0):\n    """R1 gradient penalty (Mescheder et al., 2018).\n\n    Regularizes discriminator to have small gradients on real data,\n    which prevents the discriminator from becoming too confident and\n    stabilizes GAN training.\n    """\n    real_images = real_images.detach().requires_grad_(True)\n    d_real = discriminator(real_images)\n    grad_real = autograd.grad(\n        outputs=d_real.sum(),\n        inputs=real_images,\n        create_graph=True,\n    )[0]\n    penalty = grad_real.pow(2).reshape(grad_real.shape[0], -1).sum(1).mean()\n    return weight * penalty\n\n\ndef feature_matching_loss(d_feats_fake, d_feats_real):\n    """Feature matching loss: L1 between discriminator features of fake vs real.\n\n    Matches statistics at each discriminator layer. Alignment-free because\n    it compares feature distributions, not pixel-level correspondence.\n    """\n    loss = 0.0\n    for feat_fake, feat_real in zip(d_feats_fake, d_feats_real):\n        loss += torch.nn.functional.l1_loss(feat_fake, feat_real.detach())\n    return loss / len(d_feats_fake)\n\n\nclass MultiScaleDiscriminator(nn.Module):\n    """Two PatchGAN discriminators at different scales."""\n\n    def __init__(self, in_channels=6, ndf=64, n_layers=3):\n        super().__init__()\n        self.disc_512 = PatchDiscriminator(in_channels, ndf, n_layers)\n        self.disc_256 = PatchDiscriminator(in_channels, ndf, n_layers)\n\n    def forward(self, x, return_features=False):\n        """\n        Args:\n            x: [B, 6, 512, 512] concat(output, H&E)\n\n        Returns:\n            list of (logits, [features]) from each scale\n        """\n        x_256 = F.interpolate(x, size=256, mode=\'bilinear\', align_corners=False)\n\n        if return_features:\n            out_512, feats_512 = self.disc_512(x, return_features=True)\n            out_256, feats_256 = self.disc_256(x_256, return_features=True)\n            return [(out_512, feats_512), (out_256, feats_256)]\n        else:\n            out_512 = self.disc_512(x)\n            out_256 = self.disc_256(x_256)\n            return [out_512, out_256]\n', 'asp_loss.py': 'import time\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\n\nclass AdaptiveSupervisedPatchNCELoss(nn.Module):\n\n    def __init__(self, opt):\n        super().__init__()\n        self.opt = opt\n        self.cross_entropy_loss = torch.nn.CrossEntropyLoss(reduction=\'none\')\n        self.mask_dtype = torch.bool\n        self.total_epochs = opt.n_epochs + opt.n_epochs_decay\n\n    def forward(self, feat_q, feat_k, current_epoch=-1):\n        num_patches = feat_q.shape[0]\n        dim = feat_q.shape[1]\n        feat_k = feat_k.detach()\n\n        # pos logit\n        l_pos = torch.bmm(\n            feat_q.view(num_patches, 1, -1), feat_k.view(num_patches, -1, 1))\n        l_pos = l_pos.view(num_patches, 1)\n\n        # neg logit\n\n        # Should the negatives from the other samples of a minibatch be utilized?\n        # In CUT and FastCUT, we found that it\'s best to only include negatives\n        # from the same image. Therefore, we set\n        # --nce_includes_all_negatives_from_minibatch as False\n        # However, for single-image translation, the minibatch consists of\n        # crops from the "same" high-resolution image.\n        # Therefore, we will include the negatives from the entire minibatch.\n        if self.opt.nce_includes_all_negatives_from_minibatch:\n            # reshape features as if they are all negatives of minibatch of size 1.\n            batch_dim_for_bmm = 1\n        else:\n            batch_dim_for_bmm = self.opt.batch_size\n\n        # reshape features to batch size\n        feat_q = feat_q.view(batch_dim_for_bmm, -1, dim)\n        feat_k = feat_k.view(batch_dim_for_bmm, -1, dim)\n        npatches = feat_q.size(1)\n        l_neg_curbatch = torch.bmm(feat_q, feat_k.transpose(2, 1))\n\n        # diagonal entries are similarity between same features, and hence meaningless.\n        # just fill the diagonal with very small number, which is exp(-10) and almost zero\n        diagonal = torch.eye(npatches, device=feat_q.device, dtype=self.mask_dtype)[None, :, :]\n        l_neg_curbatch.masked_fill_(diagonal, -10.0)\n        l_neg = l_neg_curbatch.view(-1, npatches)\n\n        out = torch.cat((l_pos, l_neg), dim=1) / self.opt.nce_T\n\n        loss = self.cross_entropy_loss(out, torch.zeros(out.size(0), dtype=torch.long,\n                                                        device=feat_q.device))\n\n        if self.opt.asp_loss_mode == \'none\':\n            return loss\n\n        scheduler, lookup = self.opt.asp_loss_mode.split(\'_\')[:2]\n        # Compute scheduling\n        t = (current_epoch - 1) / self.total_epochs\n        if scheduler == \'sigmoid\':\n            p = 1 / (1 + np.exp((t - 0.5) * 10))\n        elif scheduler == \'linear\':\n            p = 1 - t\n        elif scheduler == \'lambda\':\n            k = 1 - self.opt.n_epochs_decay / self.total_epochs\n            m = 1 / (1 - k)\n            p = m - m * t if t >= k else 1.0\n        elif scheduler == \'zero\':\n            p = 1.0\n        else:\n            raise ValueError(f"Unrecognized scheduler: {scheduler}")\n        # Weight lookups\n        w0 = 1.0\n        x = l_pos.squeeze().detach()\n        if lookup == \'top\':\n            x = torch.where(x > 0.0, x, torch.zeros_like(x))\n            w1 = torch.sqrt(1 - (x - 1) ** 2)\n        elif lookup == \'linear\':\n            w1 = torch.relu(x)\n        elif lookup == \'bell\':\n            sigma, mu, sc = 1, 0, 4\n            w1 = 1 / (sigma * np.sqrt(2 * torch.pi)) * torch.exp(-((x - 0.5) * sc - mu) ** 2 / (2 * sigma ** 2))\n        elif lookup == \'uniform\':\n            w1 = torch.ones_like(x)\n        else:\n            raise ValueError(f"Unrecognized lookup: {lookup}")\n        # Apply weights with schedule\n        w = p * w0 + (1 - p) * w1\n        # Normalize\n        w = w / w.sum() * len(w)\n        loss = loss * w\n        return loss\n', 'uni_losses.py': '"""\nLoss functions for UNIStainNet.\n\n- VGGFeatureExtractor: intermediate VGG16 features for Gram-matrix style loss\n- gram_matrix: compute Gram matrix of feature maps\n- PatchNCELoss: contrastive loss between H&E input and generated output (alignment-free)\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass VGGFeatureExtractor(nn.Module):\n    """Extract intermediate VGG16 features for Gram-matrix style loss.\n\n    Uses early VGG layers (relu1_2, relu2_2, relu3_3) which capture texture\n    at different scales. Gram matrices of these features are alignment-invariant\n    texture descriptors — they measure feature co-occurrence statistics, not\n    spatial layout (Gatys et al., 2016).\n    """\n\n    def __init__(self):\n        super().__init__()\n        from torchvision.models import vgg16, VGG16_Weights\n        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features\n        # Extract at relu1_2 (idx 4), relu2_2 (idx 9), relu3_3 (idx 16)\n        self.slice1 = nn.Sequential(*list(vgg.children())[:4])   # → relu1_2\n        self.slice2 = nn.Sequential(*list(vgg.children())[4:9])  # → relu2_2\n        self.slice3 = nn.Sequential(*list(vgg.children())[9:16]) # → relu3_3\n        # Freeze\n        for p in self.parameters():\n            p.requires_grad = False\n        self.eval()\n        # ImageNet normalization\n        self.register_buffer(\'mean\', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n        self.register_buffer(\'std\', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n\n    def forward(self, x):\n        """\n        Args:\n            x: [B, 3, H, W] in [-1, 1]\n        Returns:\n            list of feature maps at 3 scales\n        """\n        # Normalize: [-1,1] → [0,1] → ImageNet\n        x = (x + 1) / 2\n        x = (x - self.mean) / self.std\n        f1 = self.slice1(x)\n        f2 = self.slice2(f1)\n        f3 = self.slice3(f2)\n        return [f1, f2, f3]\n\n\ndef gram_matrix(feat):\n    """Compute Gram matrix of feature map.\n\n    Args:\n        feat: [B, C, H, W]\n    Returns:\n        gram: [B, C, C] — normalized by spatial size\n    """\n    B, C, H, W = feat.shape\n    feat_flat = feat.reshape(B, C, H * W)  # [B, C, N]\n    gram = torch.bmm(feat_flat, feat_flat.transpose(1, 2))  # [B, C, C]\n    return gram / (C * H * W)\n\n\nclass PatchNCELoss(nn.Module):\n    """Patchwise Noise Contrastive Estimation loss.\n\n    Compares H&E input and generated IHC through the generator\'s encoder.\n    For each spatial position in the generated features, the corresponding\n    position in the H&E features is the positive, and random other positions\n    are negatives. Never sees GT IHC.\n\n    Reference: Park et al., "Contrastive Learning for Unpaired Image-to-Image\n    Translation" (ECCV 2020) — adapted for paired (misaligned) setting.\n    """\n\n    def __init__(self, layer_channels, num_patches=256, temperature=0.07):\n        """\n        Args:\n            layer_channels: dict {layer_idx: channels} for each encoder layer\n            num_patches: number of spatial positions to sample per layer\n            temperature: InfoNCE temperature\n        """\n        super().__init__()\n        self.num_patches = num_patches\n        self.temperature = temperature\n\n        # 2-layer MLP projection head per encoder layer\n        self.mlps = nn.ModuleDict()\n        for layer_idx, ch in layer_channels.items():\n            self.mlps[str(layer_idx)] = nn.Sequential(\n                nn.Linear(ch, 256),\n                nn.ReLU(inplace=True),\n                nn.Linear(256, 256),\n            )\n\n    def forward(self, feats_src, feats_tgt):\n        """Compute PatchNCE loss across encoder layers.\n\n        Args:\n            feats_src: dict {layer_idx: [B, C, H, W]} from H&E input\n            feats_tgt: dict {layer_idx: [B, C, H, W]} from generated IHC\n\n        Returns:\n            scalar loss\n        """\n        total_loss = 0.0\n        n_layers = 0\n\n        for layer_idx_str, mlp in self.mlps.items():\n            layer_idx = int(layer_idx_str)\n            feat_src = feats_src[layer_idx]  # [B, C, H, W]\n            feat_tgt = feats_tgt[layer_idx]  # [B, C, H, W]\n\n            B, C, H, W = feat_src.shape\n            n_total = H * W\n\n            # Reshape to [B, C, H*W] then [B, H*W, C]\n            src_flat = feat_src.flatten(2).permute(0, 2, 1)  # [B, HW, C]\n            tgt_flat = feat_tgt.flatten(2).permute(0, 2, 1)  # [B, HW, C]\n\n            # Sample random spatial positions\n            n_sample = min(self.num_patches, n_total)\n            idx = torch.randperm(n_total, device=feat_src.device)[:n_sample]\n\n            src_sampled = src_flat[:, idx, :]  # [B, n_sample, C]\n            tgt_sampled = tgt_flat[:, idx, :]  # [B, n_sample, C]\n\n            # Project through MLP\n            src_proj = mlp(src_sampled)  # [B, n_sample, 256]\n            tgt_proj = mlp(tgt_sampled)  # [B, n_sample, 256]\n\n            # L2 normalize\n            src_proj = F.normalize(src_proj, dim=-1)\n            tgt_proj = F.normalize(tgt_proj, dim=-1)\n\n            # InfoNCE: for each query (tgt), positive is matching src position\n            # negatives are all other src positions\n            # logits: [B, n_sample, n_sample] — (i,j) = similarity of tgt_i to src_j\n            logits = torch.bmm(tgt_proj, src_proj.transpose(1, 2))  # [B, n, n]\n            logits = logits / self.temperature\n\n            # Target: diagonal (position i matches position i)\n            target = torch.arange(n_sample, device=logits.device).unsqueeze(0).expand(B, -1)\n\n            loss = F.cross_entropy(logits.flatten(0, 1), target.flatten(0, 1))\n            total_loss = total_loss + loss\n            n_layers += 1\n\n        return total_loss / n_layers if n_layers > 0 else total_loss\n', 'uni_methods.py': '"""Unmodified method bodies extracted from UNIStainNet trainer; provenance.json records source."""\nimport torch\nfrom torch.nn import functional as F\n\nclass UniLossMethods:\n    def compute_dab_intensity_loss(self, generated, target):\n            """Top-10% percentile matching for DAB intensity."""\n            with torch.amp.autocast(\'cuda\', enabled=False):\n                gen = generated.float()\n                tgt = target.float()\n    \n                dab_gen = self.dab_extractor.extract_dab_intensity(gen, normalize="none")\n                dab_tgt = self.dab_extractor.extract_dab_intensity(tgt, normalize="none")\n    \n                def _batched_top10_mean(dab):\n                    """Compute mean of top-10% DAB intensity per sample (vectorized)."""\n                    B = dab.shape[0]\n                    flat = dab.reshape(B, -1)  # [B, H*W]\n                    p99 = torch.quantile(flat, 0.99, dim=1, keepdim=True)\n                    flat = flat.clamp(max=p99)\n                    p90 = torch.quantile(flat, 0.9, dim=1, keepdim=True)\n                    mask = flat >= p90  # [B, H*W]\n                    # Use masked mean: sum(vals * mask) / sum(mask), fallback to flat mean\n                    masked_sum = (flat * mask).sum(dim=1)\n                    mask_count = mask.sum(dim=1).clamp(min=1)\n                    return masked_sum / mask_count  # [B]\n    \n                gen_scores = _batched_top10_mean(dab_gen)\n                tgt_scores = _batched_top10_mean(dab_tgt)\n                return F.l1_loss(gen_scores, tgt_scores)\n\n    def compute_he_edge_loss(self, generated, he_input):\n            """H&E edge structure preservation loss.\n    \n            Extracts Sobel edges from H&E input and generated output, then\n            computes L1 loss between edge maps at multiple scales.\n            """\n            with torch.amp.autocast(\'cuda\', enabled=False):\n                gen = generated.float()\n                he = he_input.float()\n    \n                gen_gray = ((gen + 1) / 2).mean(dim=1, keepdim=True)\n                he_gray = ((he + 1) / 2).mean(dim=1, keepdim=True)\n    \n                sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],\n                                       dtype=torch.float32, device=gen.device).view(1, 1, 3, 3)\n                sobel_y = sobel_x.transpose(-1, -2)\n    \n                loss = 0.0\n                full_size = gen_gray.shape[-1]\n                scales = [full_size, full_size // 2]\n                for size in scales:\n                    if size < full_size:\n                        g = F.interpolate(gen_gray, size=size, mode=\'bilinear\', align_corners=False)\n                        h = F.interpolate(he_gray, size=size, mode=\'bilinear\', align_corners=False)\n                    else:\n                        g, h = gen_gray, he_gray\n    \n                    gx_gen = F.conv2d(g, sobel_x, padding=1)\n                    gy_gen = F.conv2d(g, sobel_y, padding=1)\n                    edge_gen = (gx_gen**2 + gy_gen**2 + 1e-8).sqrt()\n    \n                    gx_he = F.conv2d(h, sobel_x, padding=1)\n                    gy_he = F.conv2d(h, sobel_y, padding=1)\n                    edge_he = (gx_he**2 + gy_he**2 + 1e-8).sqrt()\n    \n                    loss = loss + F.l1_loss(edge_gen, edge_he.detach())\n    \n                return loss / 2.0\n', 'uni_dab.py': '"""\nDAB (3,3\'-Diaminobenzidine) stain extraction via color deconvolution.\n\nReference: Ruifrok & Johnston, "Quantification of histochemical\nstaining by color deconvolution", Anal Quant Cytol Histol 2001\n"""\n\nimport torch\nimport torch.nn.functional as F\n\n\nclass DABExtractor:\n    """Extract DAB stain intensity from IHC images using color deconvolution.\n\n    Uses the Ruifrok & Johnston H-DAB stain matrix with softplus smoothing\n    for differentiable training loss computation.\n    """\n\n    def __init__(self, device=\'cuda\'):\n        self.device = device\n\n        # Standard H-DAB stain matrix (Ruifrok & Johnston)\n        # Each row is a stain vector in RGB optical density space\n        self.stain_matrix = torch.tensor([\n            [0.268, 0.570, 0.776],  # DAB (brown)\n            [0.650, 0.704, 0.286],  # Hematoxylin (blue)\n        ], device=device, dtype=torch.float32)\n\n        # Pseudo-inverse for deconvolution: [3, 2]\n        self.deconv_matrix = torch.linalg.pinv(self.stain_matrix.T)\n\n    def rgb_to_od(self, rgb_images: torch.Tensor) -> torch.Tensor:\n        """Convert RGB [0,1] to optical density: OD = -log10(I/I0)."""\n        rgb_images = rgb_images.clamp(1e-6, 1.0)\n        return -torch.log10(rgb_images + 1e-6)\n\n    def extract_dab_intensity(\n        self,\n        images: torch.Tensor,\n        normalize: str = "max"\n    ) -> torch.Tensor:\n        """Extract DAB stain intensity from IHC images.\n\n        Args:\n            images: [B, 3, H, W] RGB images in [-1, 1] or [0, 1]\n            normalize: "none", "max", or "meanstd"\n\n        Returns:\n            dab_intensity: [B, H, W] DAB intensity map\n        """\n        B, C, H, W = images.shape\n        assert C == 3, "Input must be RGB images"\n\n        # Auto-convert [-1, 1] -> [0, 1] if needed\n        if images.min() < 0:\n            images = (images + 1.0) / 2.0\n\n        od = self.rgb_to_od(images)\n        od_flat = od.permute(0, 2, 3, 1).reshape(-1, 3)\n\n        # Ensure deconv_matrix is on same device as input\n        deconv_matrix = self.deconv_matrix.to(od_flat.device)\n\n        # Deconvolve: concentrations = OD @ M_inv^T\n        concentrations = od_flat @ deconv_matrix.T\n        dab_flat = concentrations[:, 0]  # DAB channel\n\n        dab_intensity = dab_flat.reshape(B, H, W)\n\n        # Softplus for smooth gradients (beta=5.0 for sharper transition)\n        dab = F.softplus(dab_intensity, beta=5.0)\n\n        if normalize == "max" or normalize is True:\n            mx = dab.amax(dim=(1, 2), keepdim=True).clamp(min=1e-6)\n            dab = dab / mx\n        elif normalize == "meanstd":\n            mean = dab.mean(dim=(1, 2), keepdim=True)\n            std = dab.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)\n            dab = (dab - mean) / std\n        elif normalize == "none" or normalize is False:\n            pass\n        else:\n            raise ValueError(f"Unknown normalization: {normalize}")\n\n        return dab\n'}
CODE=WORK_ROOT/"paper-fm-code";vendor_dir=CODE/"vendor";vendor_dir.mkdir(parents=True,exist_ok=True)
for name,text in VENDOR.items():(vendor_dir/name).write_text(text)
sys.path.insert(0,str(CODE))
print("Author source provenance:");display(json.loads(VENDOR["provenance.json"]))

In [ ]:
"""Ready-library conditional FM with ASP/HistDiST/UNIStainNet-inspired supervision."""

from __future__ import annotations

import json
import random
import time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from diffusers import UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from ema_pytorch import EMA
from ihc_flow.data import PairedDataset, TrainingBatches
from ihc_flow.download import sha256_file
from ihc_flow.runtime import initial_noise, save_image, seed_everything, write_json
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchcfm.conditional_flow_matching import (
    ConditionalFlowMatcher,
    VariancePreservingConditionalFlowMatcher,
)
from torchdiffeq import odeint
from torchvision.transforms import InterpolationMode, Normalize, Resize
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, ViTConfig, ViTModel
from vendor.asp_loss import AdaptiveSupervisedPatchNCELoss
from vendor.uni_dab import DABExtractor
from vendor.uni_discriminator import (
    PatchDiscriminator,
    feature_matching_loss,
    hinge_loss_d,
    hinge_loss_g,
    r1_gradient_penalty,
)
from vendor.uni_losses import PatchNCELoss
from vendor.uni_methods import UniLossMethods


def recipe(
    smoke=False, path="linear", losses="stable", image_size=256, train_pairs=None
):
    if path not in {"linear", "vp"} or losses not in {
        "stable",
        "all",
        "uni",
        "asp",
        "fm",
    }:
        raise ValueError("Unknown experiment")
    if image_size not in {256, 512}:
        raise ValueError("Use256 or512")
    if train_pairs is not None and train_pairs < 4:
        raise ValueError("Use at least4 train pairs")
    return dict(
        method="paper_conditioned_fm_v2",
        smoke=smoke,
        path=path,
        losses=losses,
        crop=64 if smoke else 512,
        image_size=64 if smoke else image_size,
        train_pairs=16 if smoke else train_pairs,
        batch=2 if smoke else 4,
        source_sigma=0.01,
        solver="euler",
        ode_steps=2 if smoke else 8,
        total_steps=8 if smoke else 6000,
        lr=0.001 if smoke else (3e-5 if losses == "stable" else 1e-4),
        warmup=0 if smoke else 200,
        ema=0.9 if smoke else 0.995,
        aux_every=1 if smoke else (2 if losses == "stable" else 4),
        aux_start=0 if smoke else (0 if losses == "stable" else 200),
        aux_ramp=1 if smoke else (100 if losses == "stable" else 500),
        gan_start=0 if smoke else 1000,
        r1_every=4 if smoke else 16,
        r1_gamma=10.0,
        critic_enabled=losses in {"all", "uni"},
        validate_every=4 if smoke else (250 if losses == "stable" else 500),
        validation_count=4 if smoke else 8,
        seed=42,
        weights={
            "lpips": 1.0,
            "low_l1": 1.0,
            "he_edge": 0.1,
            "dab": 0.2,
            "patchnce": 0.25,
            "asp": 1.0,
            "adversarial": 0.1,
            "feature_matching": 1.0,
            "range": 0.1,
        },
    )

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display,FileLink
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if MODE not in {"train","smoke"}:raise ValueError("MODE must be train/smoke")
if MODE=="train" and DEVICE.type!="cuda":raise RuntimeError("Включи GPU или MODE='smoke'")
torch.set_num_threads(4)
CFG=recipe(MODE=="smoke",FM_PATH,LOSS_RECIPE,IMAGE_SIZE,TRAIN_PAIRS)
key=__import__('hashlib').sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:8]
RUN=WORK_ROOT/f"her2-paper-fm-v2-{FM_PATH}-{LOSS_RECIPE}-{key}"
if INITIALIZE_FROM and RESUME_FROM is None and not (RUN/"latest.pt").exists() and not Path(INITIALIZE_FROM).is_file():
    raise FileNotFoundError("Не найден INITIALIZE_FROM: подключи прежние Outputs и укажи /kaggle/input/.../best.pt; None означает новый run с нуля")
WEIGHT_CACHE=WORK_ROOT/"paper-fm-cache/torch";torch.hub.set_dir(str(WEIGHT_CACHE))
(WEIGHT_CACHE/"checkpoints").mkdir(parents=True,exist_ok=True)
for filename in ("alexnet-owt-7be5be79.pth",):
    candidates=list(INPUT_ROOT.rglob(filename))
    if candidates and not (WEIGHT_CACHE/"checkpoints"/filename).exists():
        (WEIGHT_CACHE/"checkpoints"/filename).symlink_to(candidates[0].resolve())
if PHIKON_PATH is None and MODE!="smoke":
    candidates=[p.parent for p in INPUT_ROOT.rglob("phikon_bundle.json") if (p.parent/"model.safetensors").exists()]
    if not candidates:candidates=[p.parent for p in INPUT_ROOT.rglob("config.json") if 'phikon' in str(p).lower() and (p.parent/"model.safetensors").exists()]
    if candidates:PHIKON_PATH=str(candidates[0])
    else:
        archives=list(INPUT_ROOT.rglob("phikon-kaggle.zip"))
        if len(archives)==1:
            folder=WORK_ROOT/"phikon"
            with zipfile.ZipFile(archives[0]) as archive:
                for info in archive.infolist():
                    if not (folder/info.filename).resolve().is_relative_to(folder.resolve()):raise ValueError("Unsafe model ZIP")
                archive.extractall(folder)
            PHIKON_PATH=str(folder)
audit=json.loads((MANIFEST.parent/"audit.json").read_text());assert not any(audit['group_overlap'].values())
print("Device:",DEVICE,"Split:",audit['counts'],"Run:",RUN)
display(CFG)

## Source and conditioning

Source is `H&E + 0.01 × noise`; target is real IHC. The Diffusers U-Net receives H&E and Phikon tokens. Native 512-pixel crops are resized to 256. No OT is used in this recipe.


In [ ]:
class DataPairs(PairedDataset):
    def __init__(self, *args, image_size, **kwargs):
        super().__init__(*args, **kwargs)
        self.image_size = image_size

    def __getitem__(self, index):
        item = super().__getitem__(index)
        images = F.interpolate(
            torch.stack((item["he"], item["ihc"])),
            (self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False,
            antialias=True,
        )
        item["he"], item["ihc"] = images.unbind(0)
        return item


def balanced_records(records, count, seed=42):
    count = len(records) if count is None else min(count, len(records))
    groups = {}
    rng = random.Random(seed)
    for record in records:
        groups.setdefault(record["group"], []).append(record)
    order = sorted(groups)
    rng.shuffle(order)
    for rows in groups.values():
        rng.shuffle(rows)
    result = []
    while len(result) < count:
        for key in order:
            if groups[key] and len(result) < count:
                result.append(groups[key].pop())
    return result


def validation_data(root, manifest, cfg):
    data = DataPairs(root, manifest, "val", cfg["crop"], image_size=cfg["image_size"])
    data.records = balanced_records(data.records, cfg["validation_count"], cfg["seed"])
    return data


class PhikonCondition(nn.Module):
    """The ready HF Phikon used by HistDiST; frozen, conditioning only."""

    def __init__(self, smoke=False, path=None):
        super().__init__()
        if smoke:
            self.encoder = ViTModel(
                ViTConfig(
                    image_size=64,
                    patch_size=8,
                    hidden_size=32,
                    num_hidden_layers=1,
                    num_attention_heads=4,
                    intermediate_size=64,
                ),
                add_pooling_layer=False,
            )
            self.size, self.dimension, self.identity = (
                64,
                32,
                "random-smoke-phikon-shape-only",
            )
            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        else:
            if path is None:
                from huggingface_hub import snapshot_download

                path = snapshot_download(
                    "owkin/phikon",
                    revision="057cc0295895c2df3dd7681a89680da6015cbefe",
                    allow_patterns=[
                        "config.json",
                        "preprocessor_config.json",
                        "model.safetensors",
                    ],
                )
            path = Path(path)
            processor = AutoImageProcessor.from_pretrained(path, local_files_only=True)
            self.encoder = ViTModel.from_pretrained(
                path, add_pooling_layer=False, local_files_only=True
            )
            self.size, self.dimension = 224, 768
            mean, std = processor.image_mean, processor.image_std
            import hashlib

            h = hashlib.sha256()
            for name in (
                "config.json",
                "preprocessor_config.json",
                "model.safetensors",
            ):
                h.update(bytes.fromhex(sha256_file(path / name)))
            self.identity = h.hexdigest()
        self.resize = Resize(
            (self.size, self.size), InterpolationMode.BICUBIC, antialias=True
        )
        self.normalize = Normalize(mean, std)
        self.eval().requires_grad_(False)

    def train(self, mode=True):
        return super().train(False)

    @torch.no_grad()
    def forward(self, he):
        return self.encoder(
            pixel_values=self.normalize(self.resize((he + 1) / 2))
        ).last_hidden_state


class FlowModel(nn.Module):
    """A small adapter around the ready Diffusers conditional U-Net; no custom U-Net."""

    def __init__(self, cfg, context_dimension):
        super().__init__()
        if cfg["smoke"]:
            channels = (32, 64)
            down = ("DownBlock2D", "CrossAttnDownBlock2D")
            up = ("CrossAttnUpBlock2D", "UpBlock2D")
        else:
            channels = (32, 64, 128, 128)
            down = (
                "DownBlock2D",
                "DownBlock2D",
                "CrossAttnDownBlock2D",
                "CrossAttnDownBlock2D",
            )
            up = ("CrossAttnUpBlock2D", "CrossAttnUpBlock2D", "UpBlock2D", "UpBlock2D")
        self.unet = UNet2DConditionModel(
            sample_size=cfg["image_size"],
            in_channels=6,
            out_channels=3,
            block_out_channels=channels,
            down_block_types=down,
            up_block_types=up,
            layers_per_block=1 if cfg["smoke"] else 2,
            norm_num_groups=8,
            cross_attention_dim=context_dimension,
            attention_head_dim=4 if cfg["smoke"] else 8,
            only_cross_attention=True,
        )

    def forward(self, state, time_point, he, tokens):
        return self.unet(
            torch.cat((state, he), 1), time_point * 1000, encoder_hidden_states=tokens
        ).sample.float()


def make_matcher(cfg):
    cls = (
        ConditionalFlowMatcher
        if cfg["path"] == "linear"
        else VariancePreservingConditionalFlowMatcher
    )
    return cls(sigma=0.0)


def integrate(model, he, tokens, source, steps=8, solver="euler"):
    """Ready torchdiffeq integration, also used for differentiable auxiliary endpoints."""
    if steps < 1 or solver not in {"euler", "heun2", "rk4"}:
        raise ValueError("Unknown fixed-step solver")

    def field(t, state):
        return model(state, t.expand(len(state)), he, tokens)

    return odeint(
        field,
        source,
        torch.tensor([0.0, 1.0], device=he.device),
        method=solver,
        options={"step_size": 1 / steps},
    )[-1]


@torch.inference_mode()
def sample(model, phikon, he, identities, cfg, seed=42, steps=None, solver=None):
    if model.training:
        raise ValueError("Use eval()")
    with torch.autocast(
        he.device.type, dtype=torch.float16, enabled=he.device.type == "cuda"
    ):
        tokens = phikon(he)
        source = he + cfg["source_sigma"] * initial_noise(he, identities, seed)
        return integrate(
            model,
            he,
            tokens,
            source,
            steps or cfg["ode_steps"],
            solver or cfg["solver"],
        )

## Supervision on generated images

Auxiliary losses use a real Euler8 rollout, without the IHC target as a model input. The stable recipe disables GAN, ASP, PatchNCE and H&E Sobel losses.


In [ ]:
class PaperLosses(nn.Module):
    """Author losses with small adapters for FM endpoints, frozen features and RGB range."""

    def __init__(self, cfg, device):
        super().__init__()
        self.cfg = cfg
        self.use_uni = cfg["losses"] in {"stable", "all", "uni"}
        self.use_asp = cfg["losses"] in {"all", "asp"}
        self.dab = DABExtractor(device="cpu")
        self.methods = UniLossMethods()
        self.methods.dab_extractor = self.dab
        self.projector = None
        self.perceptual = None
        self.features = None
        if self.use_uni or self.use_asp:
            import lpips

            self.perceptual = (
                lpips.LPIPS(net="alex", pnet_rand=cfg["smoke"], verbose=False)
                .eval()
                .requires_grad_(False)
            )
        if self.use_asp:
            # Reuse frozen pretrained LPIPS AlexNet features for both contrastive losses.
            self.projector = PatchNCELoss(
                {0: 64, 1: 192, 2: 384},
                num_patches=32 if cfg["smoke"] else 128,
                temperature=0.07,
            )
            self.asp_options = SimpleNamespace(
                n_epochs=10,
                n_epochs_decay=90,
                batch_size=1,
                nce_includes_all_negatives_from_minibatch=False,
                nce_T=0.07,
                asp_loss_mode="lambda_linear",
            )
            self.asp = AdaptiveSupervisedPatchNCELoss(self.asp_options)
        self.to(device)

    def trainable_parameters(self):
        return list(self.projector.parameters()) if self.projector is not None else []

    def feature_maps(self, image):
        image = F.interpolate(
            image,
            (64 if self.cfg["smoke"] else 128,) * 2,
            mode="bilinear",
            align_corners=False,
        )
        maps = self.perceptual.net(self.perceptual.scaling_layer(image))
        return {i: value for i, value in enumerate(maps[:3])}

    def dab_score(self, image):
        # Explicit [0,1] avoids the author's ambiguous auto-range detection on all-positive [-1,1] images.
        rgb = ((image.float() + 1) / 2).clamp(1e-3, 1)
        dab = self.dab.extract_dab_intensity(rgb, normalize="none").flatten(1)
        limit = torch.quantile(dab, 0.99, dim=1, keepdim=True)
        dab = dab.clamp(max=limit)
        threshold = torch.quantile(dab, 0.9, dim=1, keepdim=True)
        mask = dab >= threshold
        return (dab * mask).sum(1) / mask.sum(1).clamp_min(1)

    def generator_terms(self, prediction, he, target, step, discriminator, adversarial):
        image = prediction.clamp(-1, 1)
        terms = {"range": F.relu(prediction.abs() - 1).square().mean()}
        if self.use_uni:
            resolutions = (64, 64) if self.cfg["smoke"] else (128, 256)
            a, b = (
                F.interpolate(
                    x, (resolutions[0],) * 2, mode="bilinear", align_corners=False
                )
                for x in (image, target)
            )
            perceptual = self.perceptual(a.float(), b.float()).mean()
            if resolutions[1] != resolutions[0]:
                a, b = (
                    F.interpolate(
                        x, (resolutions[1],) * 2, mode="bilinear", align_corners=False
                    )
                    for x in (image, target)
                )
                perceptual = (
                    perceptual + 0.5 * self.perceptual(a.float(), b.float()).mean()
                )
            terms["lpips"] = perceptual
            small = min(64, image.shape[-1])
            a, b = (
                F.interpolate(x, (small, small), mode="bilinear", align_corners=False)
                for x in (image, target)
            )
            terms["low_l1"] = F.l1_loss(a, b)
            if self.cfg["losses"] != "stable":
                terms["he_edge"] = self.methods.compute_he_edge_loss(image, he)
            # Reuse the author loss, supplying explicit RGB to avoid range guessing.
            a, b = (((x.float() + 1) / 2).clamp(1e-3, 1) for x in (image, target))
            terms["dab"] = self.methods.compute_dab_intensity_loss(a, b)
        if self.use_asp:
            with torch.no_grad():
                he_features = self.feature_maps(he)
                target_features = self.feature_maps(target)
            generated_features = self.feature_maps(image)
            terms["patchnce"] = self.projector(he_features, generated_features)
            self.asp_options.batch_size = len(image)
            epoch = 1 + min(1.0, step / self.cfg["total_steps"]) * 99
            values = []
            for layer, projection in self.projector.mlps.items():
                index = int(layer)
                query = generated_features[index].flatten(2).transpose(1, 2)
                key = target_features[index].flatten(2).transpose(1, 2)
                ids = torch.randperm(query.shape[1], device=query.device)[
                    : self.projector.num_patches
                ]
                q = (
                    F.normalize(projection(query[:, ids]), dim=-1)
                    .flatten(0, 1)
                    .contiguous()
                )
                with torch.no_grad():
                    k = (
                        F.normalize(projection(key[:, ids]), dim=-1)
                        .flatten(0, 1)
                        .contiguous()
                    )
                values.append(
                    self.asp(q.float(), k.float(), current_epoch=epoch).mean()
                )
            terms["asp"] = torch.stack(values).mean()
        if adversarial:
            logits, features = discriminator(image.float(), return_features=True)
            with torch.no_grad():
                _, reference = discriminator(target.float(), return_features=True)
            terms["adversarial"] = hinge_loss_g(logits)
            # The author function is spatial L1. Pool first so our adaptation compares statistics.
            terms["feature_matching"] = feature_matching_loss(
                [a.mean((2, 3), keepdim=True) for a in features],
                [a.mean((2, 3), keepdim=True) for a in reference],
            )
        return terms

In [ ]:
def make_discriminator(cfg):
    return PatchDiscriminator(in_channels=3, ndf=8 if cfg["smoke"] else 32, n_layers=3)


def rgb(image):
    return (
        ((image.detach().float().cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy()
    )


def pixel_scores(prediction, target):
    from skimage.metrics import structural_similarity

    a, b = rgb(prediction), rgb(target)
    return dict(
        rgb_mse=float(np.square(a - b).mean()),
        ssim=float(structural_similarity(a, b, channel_axis=-1, data_range=1)),
    )


@torch.inference_mode()
def validate(model, phikon, paper_losses, data, device, cfg, directory):
    model.eval()
    rows = []
    for index in range(len(data)):
        item = data[index]
        he = item["he"].unsqueeze(0).to(device)
        target = item["ihc"].unsqueeze(0).to(device)
        prediction = sample(model, phikon, he, [item["id"]], cfg)
        row = dict(
            id=item["id"], group=item["group"], **pixel_scores(prediction[0], target[0])
        )
        with torch.autocast(device.type, enabled=False):
            row["dab_error"] = (
                (
                    paper_losses.dab_score(prediction.clamp(-1, 1))
                    - paper_losses.dab_score(target)
                )
                .abs()
                .item()
            )
            if paper_losses.perceptual is not None:
                a, b = (
                    F.interpolate(
                        x,
                        (64 if cfg["smoke"] else 128,) * 2,
                        mode="bilinear",
                        align_corners=False,
                    )
                    for x in (prediction.clamp(-1, 1), target)
                )
                row["lpips"] = (
                    paper_losses.perceptual(a.float(), b.float()).mean().item()
                )
        rows.append(row)
        if index < 4:
            save_image(
                torch.cat((item["he"], prediction[0].cpu(), item["ihc"]), 2),
                directory / f"example-{index}.png",
            )
    write_json(directory / "metrics.json", rows)
    return {
        "val_" + key: float(np.mean([row[key] for row in rows]))
        for key in ("ssim", "rgb_mse", "dab_error", "lpips")
        if key in rows[0]
    }

## Training and model selection

The stable recipe uses AdamW at 3e-5 and Euler8. Checkpoints use validation LPIPS plus DAB error on eight crops. Optional initialization loads EMA weights and resets the optimizer; strict resume restores the same recipe. Saved configuration takes precedence over run instructions.


In [ ]:
def optimizer_step(optimizer, scaler, parameters):
    scaler.unscale_(optimizer)
    finite = (
        torch.stack(
            [torch.isfinite(p.grad).all() for p in parameters if p.grad is not None]
        )
        .all()
        .item()
    )
    if finite:
        torch.nn.utils.clip_grad_norm_(parameters, 1.0, error_if_nonfinite=True)
    elif not scaler.is_enabled():
        raise FloatingPointError("Non-finite gradients")
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    return finite


def save_checkpoint(path, payload):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def train_session(
    root,
    manifest,
    directory,
    device,
    cfg,
    session_steps=2000,
    micro_batch=1,
    resume=None,
    phikon_path=None,
    progress=True,
    initialize=None,
):
    if resume is not None and initialize is not None:
        raise ValueError("Choose strict resume OR weights-only initialization")
    if session_steps < 0 or micro_batch < 1:
        raise ValueError("Invalid session/microbatch")
    if session_steps == 0 and resume is None and initialize is None:
        raise ValueError("Inference-only requires checkpoint")
    if resume is None and directory.exists() and any(directory.iterdir()):
        raise FileExistsError("Use resume or a fresh run")
    seed_everything(cfg["seed"])
    phikon = PhikonCondition(cfg["smoke"], phikon_path).to(device)
    model = FlowModel(cfg, phikon.dimension).to(device)
    ema = EMA(
        model,
        beta=cfg["ema"],
        update_after_step=0,
        update_every=1,
        include_online_model=False,
    )
    objective = PaperLosses(cfg, device)
    discriminator = make_discriminator(cfg).to(device)
    parameters = list(model.parameters()) + objective.trainable_parameters()
    optimizer = torch.optim.AdamW(parameters, lr=cfg["lr"], weight_decay=0.01)
    disc_optimizer = torch.optim.Adam(
        discriminator.parameters(), lr=cfg["lr"], betas=(0.0, 0.999)
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, cfg["warmup"], cfg["total_steps"]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    matcher = make_matcher(cfg)
    fingerprint = sha256_file(manifest)
    step, batches_seen, disc_updates, skipped, best = 0, 0, 0, 0, float("inf")
    initialized_from = None
    if resume is not None:
        state = torch.load(resume, map_location="cpu", weights_only=True)
        if (
            state.get("format") != "paper_conditioned_fm_v2"
            or state["recipe"] != cfg
            or state["manifest_sha256"] != fingerprint
            or state["phikon_sha256"] != phikon.identity
        ):
            raise ValueError("Requires the same paper-FM recipe, split and backbone")
        initialized_from = state.get("initialized_from")
        model.load_state_dict(state["model"])
        ema.load_state_dict(state["ema"])
        if objective.projector is not None:
            objective.projector.load_state_dict(state["projector"])
        discriminator.load_state_dict(state["discriminator"])
        optimizer.load_state_dict(state["optimizer"])
        disc_optimizer.load_state_dict(state["disc_optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        step, batches_seen, disc_updates, skipped, best = (
            state[k]
            for k in (
                "step",
                "batches_seen",
                "disc_updates",
                "amp_skipped",
                "best_score",
            )
        )
        torch.set_rng_state(state["torch_rng"])
        if device.type == "cuda" and state["cuda_rng"] is not None:
            torch.cuda.set_rng_state_all(state["cuda_rng"][: torch.cuda.device_count()])
    if initialize is not None:
        initial = torch.load(initialize, map_location="cpu", weights_only=True)
        if initial.get("format") not in {
            "paper_conditioned_fm_v1",
            "paper_conditioned_fm_v2",
        }:
            raise ValueError("Initialize expects paper-FM v1/v2 checkpoint")
        if (
            initial["manifest_sha256"] != fingerprint
            or initial["phikon_sha256"] != phikon.identity
        ):
            raise ValueError("Initialization split/backbone differs")
        for key in ("path", "image_size", "crop", "source_sigma"):
            if initial["recipe"][key] != cfg[key]:
                raise ValueError(f"Initialization requires matching {key}")
        weights = {
            key.removeprefix("ema_model."): value
            for key, value in initial["ema"].items()
            if key.startswith("ema_model.")
        }
        model.load_state_dict(weights, strict=True)
        ema.ema_model.load_state_dict(weights, strict=True)
        initialized_from = dict(
            filename=Path(initialize).name,
            sha256=sha256_file(initialize),
            step=initial["step"],
            weights="ema",
        )
        print(
            f"Initialized EMA from step {initial['step']}; optimizer/counter are new",
            flush=True,
        )
    if resume is None and cfg["critic_enabled"]:
        # Spectral-norm power buffers must be initialized before the first eval-mode G loss.
        discriminator.train()
        with torch.no_grad():
            dummy = torch.zeros(
                1, 3, cfg["image_size"], cfg["image_size"], device=device
            )
            for _ in range(8):
                discriminator(dummy)
        discriminator.eval()
    directory.mkdir(parents=True, exist_ok=True)
    data = DataPairs(
        root, manifest, "train", cfg["crop"], True, image_size=cfg["image_size"]
    )
    data.records = balanced_records(data.records, cfg["train_pairs"], cfg["seed"])
    validation = validation_data(root, manifest, cfg)
    if {r["group"] for r in data.records} & {r["group"] for r in validation.records}:
        raise ValueError("Train/validation group overlap")
    loader = DataLoader(
        data,
        batch_sampler=TrainingBatches(
            len(data), cfg["batch"], cfg["seed"], batches_seen
        ),
        num_workers=2 if device.type == "cuda" else 0,
        pin_memory=device.type == "cuda",
        generator=torch.Generator().manual_seed(cfg["seed"] + 1),
    )
    iterator = iter(loader)
    stop = min(cfg["total_steps"], step + session_steps)

    def payload():
        return dict(
            format="paper_conditioned_fm_v2",
            initialized_from=initialized_from,
            recipe=cfg,
            manifest_sha256=fingerprint,
            phikon_sha256=phikon.identity,
            model=model.state_dict(),
            ema=ema.state_dict(),
            projector=objective.projector.state_dict() if objective.projector else None,
            discriminator=discriminator.state_dict(),
            optimizer=optimizer.state_dict(),
            disc_optimizer=disc_optimizer.state_dict(),
            scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            step=step,
            batches_seen=batches_seen,
            disc_updates=disc_updates,
            amp_skipped=skipped,
            best_score=best,
            torch_rng=torch.get_rng_state(),
            cuda_rng=torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
        )

    write_json(directory / "recipe.json", cfg)
    write_json(directory / "initialization.json", initialized_from)
    if resume is None and initialize is not None:
        initial_metrics = validate(
            ema.ema_model,
            phikon,
            objective,
            validation,
            device,
            cfg,
            directory / "validation" / "0",
        )
        best = (
            initial_metrics.get("val_lpips", initial_metrics["val_rgb_mse"])
            + initial_metrics["val_dab_error"]
        )
        baseline = dict(
            step=0,
            event="initial_validation",
            source_step=initialized_from["step"],
            **initial_metrics,
            **{"raw_" + key: value for key, value in initial_metrics.items()},
        )
        with (directory / "history.jsonl").open("a") as stream:
            stream.write(json.dumps(baseline) + "\n")
        save_checkpoint(directory / "best.pt", payload())
        save_checkpoint(directory / "latest.pt", payload())
        print(json.dumps(baseline), flush=True)
    write_json(directory / "train-subset.json", [r["id"] for r in data.records])
    bar = tqdm(
        total=stop, initial=step, desc="Paper-loss FM updates", disable=not progress
    )
    failures = 0
    log_window = []
    while step < stop:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        batch = next(iterator)
        batches_seen += 1
        he, ihc = batch["he"], batch["ihc"]
        source = he + cfg["source_sigma"] * torch.randn_like(he)
        t, point, velocity = matcher.sample_location_and_conditional_flow(source, ihc)
        auxiliary = (
            cfg["losses"] != "fm"
            and step >= cfg["aux_start"]
            and step % cfg["aux_every"] == 0
        )
        adversarial = cfg["critic_enabled"] and auxiliary and step >= cfg["gan_start"]
        discriminator.eval().requires_grad_(False)
        values = {}
        fake_parts = []
        real_parts = []
        started = time.perf_counter()
        for first in range(0, len(he), micro_batch):
            section = slice(first, first + micro_batch)
            c, y, x0, xt, ut, time_point = (
                v[section].to(device) for v in (he, ihc, source, point, velocity, t)
            )
            weight = len(c) / len(he)
            with torch.autocast(
                device.type, dtype=torch.float16, enabled=scaler.is_enabled()
            ):
                tokens = phikon(c)
                predicted = model(xt, time_point, c, tokens)
                fm = F.mse_loss(predicted.float(), ut.float())
            if not torch.isfinite(fm):
                raise FloatingPointError("Non-finite FM loss")
            scaler.scale(fm * weight).backward()
            values["fm"] = values.get("fm", 0.0) + fm.detach().item() * weight
            if auxiliary:
                with torch.autocast(
                    device.type, dtype=torch.float16, enabled=scaler.is_enabled()
                ):
                    generated = integrate(
                        model, c, tokens, x0, cfg["ode_steps"], cfg["solver"]
                    )
                # Author perceptual/contrastive and DAB functions stay FP32 for stability.
                with torch.autocast(device.type, enabled=False):
                    terms = objective.generator_terms(
                        generated.float(),
                        c.float(),
                        y.float(),
                        step,
                        discriminator,
                        adversarial,
                    )
                    aux = sum(
                        cfg["weights"][name] * loss for name, loss in terms.items()
                    )
                ramp = min(1.0, (step - cfg["aux_start"] + 1) / cfg["aux_ramp"])
                if not torch.isfinite(aux):
                    raise FloatingPointError("Non-finite endpoint loss")
                scaler.scale(aux * ramp * weight).backward()
                for name, loss in terms.items():
                    values[name] = values.get(name, 0.0) + loss.detach().item() * weight
                if adversarial:
                    fake_parts.append(generated.detach().clamp(-1, 1))
                    real_parts.append(y.detach())
        if not optimizer_step(optimizer, scaler, parameters):
            skipped += 1
            failures += 1
            if failures >= 16:
                save_checkpoint(directory / "last-safe.pt", payload())
                raise FloatingPointError("16 AMP overflows")
            continue
        failures = 0
        scheduler.step()
        ema.update()
        if adversarial:
            discriminator.train().requires_grad_(True)
            disc_optimizer.zero_grad(set_to_none=True)
            fake, real = torch.cat(fake_parts).float(), torch.cat(real_parts).float()
            with torch.autocast(device.type, enabled=False):
                real_logits = discriminator(real)
                fake_logits = discriminator(fake)
                d_hinge = hinge_loss_d(real_logits, fake_logits)
                d_r1 = torch.zeros((), device=device)
                if disc_updates % cfg["r1_every"] == 0:
                    # R1 uses one MEAN patch score per image, consistent across patch-map sizes.
                    def mean_patch_score(images):
                        return discriminator(images).flatten(1).mean(1)

                    d_r1 = r1_gradient_penalty(
                        mean_patch_score, real, weight=cfg["r1_gamma"] / 2
                    )
                d_r1_weighted = d_r1 * cfg["r1_every"]
                disc_loss = d_hinge + d_r1_weighted
                values.update(
                    d_hinge=d_hinge.detach().item(),
                    d_r1=d_r1.detach().item(),
                    d_r1_weighted=d_r1_weighted.detach().item(),
                )
                disc_statistics = dict(
                    d_real_mean=real_logits.detach().mean().item(),
                    d_fake_mean=fake_logits.detach().mean().item(),
                )
            if not torch.isfinite(disc_loss):
                raise FloatingPointError("Non-finite discriminator loss")
            disc_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                discriminator.parameters(), 1.0, error_if_nonfinite=True
            )
            disc_optimizer.step()
            disc_updates += 1
            values["discriminator"] = disc_loss.detach().item()
        step += 1
        record = dict(
            step=step,
            learning_rate=scheduler.get_last_lr()[0],
            amp_skipped=skipped,
            auxiliary=auxiliary,
            adversarial=adversarial,
            disc_updates=disc_updates,
            critic_enabled=cfg["critic_enabled"],
            seconds_per_update=time.perf_counter() - started,
            **{"loss_" + key: value for key, value in values.items()},
        )
        if adversarial:
            record.update(disc_statistics)
        ramp = min(1.0, max(0.0, (step - cfg["aux_start"]) / cfg["aux_ramp"]))
        record["loss_total"] = values["fm"] + ramp * sum(
            cfg["weights"][key] * value
            for key, value in values.items()
            if key in cfg["weights"]
        )
        if step % cfg["validate_every"] == 0 or step == stop:
            record.update(
                validate(
                    ema.ema_model,
                    phikon,
                    objective,
                    validation,
                    device,
                    cfg,
                    directory / "validation" / str(step),
                )
            )
            raw = validate(
                model,
                phikon,
                objective,
                validation,
                device,
                cfg,
                directory / "validation-raw" / str(step),
            )
            record.update({"raw_" + key: value for key, value in raw.items()})
            score = (
                record.get("val_lpips", record["val_rgb_mse"]) + record["val_dab_error"]
            )
            if score < best:
                best = score
                save_checkpoint(directory / "best.pt", payload())
        if step % 200 == 0 or step == stop:
            save_checkpoint(directory / "latest.pt", payload())
        with (directory / "history.jsonl").open("a") as stream:
            stream.write(json.dumps(record) + "\n")
        bar.update(1)
        bar.set_postfix(fm=f"{values['fm']:.4f}", refresh=False)
        log_window.append(record)
        log_window = log_window[-100:]
        if step % 100 == 0 or step == stop:
            summary = dict(record)
            keys = {key for row in log_window for key in row if key.startswith("loss_")}
            for key in keys:
                summary[key + "_mean"] = float(
                    np.mean([row[key] for row in log_window if key in row])
                )
            summary["seconds_per_update_mean"] = float(
                np.mean([row["seconds_per_update"] for row in log_window])
            )
            print(json.dumps(summary), flush=True)
    bar.close()
    return (
        (directory / "latest.pt")
        if (directory / "latest.pt").exists()
        else Path(resume)
    )

In [ ]:
resume=Path(RESUME_FROM) if RESUME_FROM else None
if resume is None and (RUN/"latest.pt").exists():resume=RUN/"latest.pt"
if resume is not None and resume.parent.resolve()!=RUN.resolve():
    RUN.mkdir(parents=True,exist_ok=True)
    for name in ("history.jsonl","best.pt"):
        if (resume.parent/name).exists() and not (RUN/name).exists():shutil.copy2(resume.parent/name,RUN/name)
initialize=Path(INITIALIZE_FROM) if INITIALIZE_FROM and resume is None else None
if resume is not None:print("Строгий v2 resume; INITIALIZE_FROM повторно не применяется")
checkpoint=train_session(DATA_ROOT,MANIFEST,RUN,DEVICE,CFG,SESSION_STEPS,MICRO_BATCH,resume,PHIKON_PATH,
                         initialize=initialize)
print("Checkpoint:",checkpoint)

## Validation gallery

Compare raw and EMA outputs across tissue groups and staining levels. Smoke mode checks execution with random feature encoders; it does not assess quality. The manifest defines the split, not train/test prefixes in IDs.


In [ ]:
history=[json.loads(line) for line in (RUN/"history.jsonl").read_text().splitlines()] if (RUN/"history.jsonl").exists() else []
fig,axes=plt.subplots(1,3,figsize=(15,4))
for name in ("fm","lpips","dab","asp","patchnce","adversarial"):
    rows=[r for r in history if "loss_"+name in r]
    axes[0].plot([r['step'] for r in rows],[r['loss_'+name] for r in rows],label=name)
axes[0].legend();axes[0].set_title('Loss components; absent aux != zero')
rows=[r for r in history if 'val_ssim' in r]
axes[1].plot([r['step'] for r in rows],[r['val_ssim'] for r in rows],'o-',label='EMA')
axes[1].plot([r['step'] for r in rows],[r['raw_val_ssim'] for r in rows],'o--',label='raw');axes[1].legend();axes[1].set_title('Validation SSIM')
for name in ('val_lpips','val_dab_error'):
    rows=[r for r in history if name in r];axes[2].plot([r['step'] for r in rows],[r[name] for r in rows],label=name)
axes[2].legend();fig.tight_layout();RUN.mkdir(parents=True,exist_ok=True);fig.savefig(RUN/'curves.png',dpi=140);plt.show();plt.close(fig)
state=torch.load(checkpoint,map_location='cpu',weights_only=True)
phikon=PhikonCondition(CFG['smoke'],PHIKON_PATH).to(DEVICE)
model=FlowModel(CFG,phikon.dimension).to(DEVICE).eval().requires_grad_(False)
ema_weights={key.removeprefix('ema_model.'):value for key,value in state['ema'].items() if key.startswith('ema_model.')}
validation=validation_data(DATA_ROOT,MANIFEST,CFG)
OUTPUT=WORK_ROOT/'paper-fm-preview'/RUN.name/f"step-{state['step']}";OUTPUT.mkdir(parents=True,exist_ok=True)
predictions={};metrics=[]
for label,weights in [('model',state['model']),('ema',ema_weights)]:
    model.load_state_dict(weights,strict=True);predictions[label]=[]
    for index in range(len(validation)):
        item=validation[index];he=item['he'].unsqueeze(0).to(DEVICE)
        result=sample(model,phikon,he,[item['id']],CFG)[0].cpu()
        predictions[label].append(result);save_image(result,OUTPUT/'predictions'/label/f"{item['id']}.png")
        metrics.append(dict(id=item['id'],weights=label,**pixel_scores(result,item['ihc'])))
for first in range(0,len(validation),3):
    count=min(3,len(validation)-first);fig,axes=plt.subplots(count,4,figsize=(14,3.5*count),squeeze=False)
    for row,index in enumerate(range(first,first+count)):
        item=validation[index]
        for ax,image,title in zip(axes[row],(item['he'],item['ihc'],predictions['model'][index],predictions['ema'][index]),('H&E','Real IHC','Raw','EMA')):
            ax.imshow(rgb(image));ax.set_title(title);ax.axis('off')
    fig.tight_layout();fig.savefig(OUTPUT/f'gallery-{first//3}.png',dpi=140);plt.show();plt.close(fig)
write_json(OUTPUT/'metrics.json',metrics)
model.load_state_dict(ema_weights)

## Condition swaps and solver comparison

Check geometry as well as color when changing H&E. More solver steps need not improve a model supervised with Euler8. The saved solver comparison uses one validation example.


In [ ]:
item=validation[0];other=next(validation[i] for i in range(1,len(validation)) if validation[i]['group']!=item['group'])
he=item['he'].unsqueeze(0).to(DEVICE);changed=other['he'].unsqueeze(0).to(DEVICE)
original=sample(model,phikon,he,[item['id']],CFG)
swapped=sample(model,phikon,changed,[item['id']],CFG)
reseeded=sample(model,phikon,he,[item['id']],CFG,seed=43)
fig,axes=plt.subplots(1,5,figsize=(17,3.5))
for ax,image,title in zip(axes,(item['he'],other['he'],original[0],swapped[0],reseeded[0]),('Original H&E','Different H&E','Original result','Same noise; new H&E','Same H&E; new seed')):
    ax.imshow(rgb(image));ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(OUTPUT/'condition-seed-swap.png',dpi=140);plt.show();plt.close(fig)
write_json(OUTPUT/'condition-swap.json',dict(original_id=item['id'],changed_id=other['id'],
    condition_rgb_mae=float(np.abs(rgb(original[0])-rgb(swapped[0])).mean()),seed_rgb_mae=float(np.abs(rgb(original[0])-rgb(reseeded[0])).mean())))
variants=[('euler',CFG['ode_steps']),('heun2',CFG['ode_steps'])]
rows=[]
for solver,steps in variants:
    sample(model,phikon,he,[item['id']],CFG,solver=solver,steps=steps)
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    started=time.perf_counter();result=sample(model,phikon,he,[item['id']],CFG,solver=solver,steps=steps)[0].cpu()
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    rows.append(dict(solver=solver,steps=steps,nfe=steps*(2 if solver=='heun2' else 1),seconds=time.perf_counter()-started,**pixel_scores(result,item['ihc'])))
write_json(OUTPUT/'solver-ablation.json',rows);display(rows)

## Full-test export

Export is optional and must be run separately after validation selection. Real and generated images need the same crop/resize protocol. The group-disjoint split and resize256 differ from published MIST protocols.


In [ ]:
@torch.inference_mode()
def export_split(split='test',weights='ema'):
    if split not in {'val','test'} or weights not in {'ema','model'}:raise ValueError('Invalid export')
    data=DataPairs(DATA_ROOT,MANIFEST,split,CFG['crop'],image_size=CFG['image_size'],limit=4 if CFG['smoke'] else None)
    model.load_state_dict(ema_weights if weights=='ema' else state['model'])
    destination=OUTPUT/f'export-{split}-{weights}';rows=[]
    for index in tqdm(range(len(data)),desc=f'Export {split}'):
        item=data[index];he=item['he'].unsqueeze(0).to(DEVICE)
        result=sample(model,phikon,he,[item['id']],CFG)[0].cpu()
        save_image(result,destination/'generated'/f"{item['id']}.png")
        save_image(item['ihc'],destination/'real'/f"{item['id']}.png")
        rows.append(dict(id=item['id'],group=item['group'],**pixel_scores(result,item['ihc'])))
    write_json(destination/'per-image.json',rows)
    write_json(destination/'protocol.json',dict(count=len(data),split=split,weights=weights,
        crop=CFG['crop'],resize=CFG['image_size'],solver=CFG['solver'],steps=CFG['ode_steps'],
        manifest_sha256=sha256_file(MANIFEST),checkpoint_sha256=sha256_file(checkpoint)))
    return destination

def feature_metrics(destination):
    if importlib.util.find_spec('torch_fidelity') is None:
        subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','--target',str(DEPS),'torch-fidelity==0.3.0'],check=True)
        sys.path.insert(0,str(DEPS));importlib.invalidate_caches()
    import torch_fidelity
    destination=Path(destination);count=len(list((destination/'generated').rglob('*.png')))
    if count<2:raise ValueError('FID/KID require at least2 images')
    metrics=torch_fidelity.calculate_metrics(input1=str(destination/'generated'),input2=str(destination/'real'),
        cuda=DEVICE.type=='cuda',fid=True,kid=True,isc=False,kid_subsets=100,kid_subset_size=min(100,count),samples_find_deep=True,verbose=False)
    write_json(destination/'feature-metrics.json',metrics);return metrics
# exported=export_split()
# feature_metrics(exported)

## Save and resume

Save checkpoints, recipe, selected IDs, source provenance and previews. Resume requires matching split, recipe and frozen backbone weights.


In [ ]:
archive_path=WORK_ROOT/f'{RUN.name}-step-{state["step"]}.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for directory in (RUN,OUTPUT,CODE):
        for path in sorted(directory.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:archive.write(path,path.relative_to(WORK_ROOT))
print('Run:',RUN,'Preview:',OUTPUT);display(FileLink(str(archive_path)))